In [ ]:
import pandas as pd

DATASET_PATH = "../data/bus_anomaly_dataset.csv"
DAY_COL = "day"                  
BUS_ID_COL = "bus_id"
IS_ANOMALY_COL = "is_anomaly"
ANOMALY_TYPE_COL = "anomaly_type"

df = pd.read_csv(DATASET_PATH)
df = df.sort_values([BUS_ID_COL, DAY_COL, "timestamp"]).reset_index(drop=True)

# Group baris anomali jadi event pakai index absolut 
anomaly_df = df[df[IS_ANOMALY_COL] == 1].copy()
anomaly_df['row_gap'] = anomaly_df.index.to_series().diff()
anomaly_df['new_event'] = (anomaly_df['row_gap'] != 1) | (anomaly_df['row_gap'].isna())
anomaly_df['event_id'] = anomaly_df['new_event'].cumsum()

events = anomaly_df.groupby('event_id').agg(
    day_start=(DAY_COL, 'first'),
    day_end=(DAY_COL, 'last'),
    bus_id=(BUS_ID_COL, 'first'),
    anomaly_type=(ANOMALY_TYPE_COL, 'first'),
    n_rows=('event_id', 'count')
).reset_index()

print(events[events['day_start'] != events['day_end']])

print(events['day_start'].value_counts().sort_index())

TEST_DAY_START = 25
test_window = events[events['day_start'] >= TEST_DAY_START]
print(test_window['anomaly_type'].value_counts())

missing_types = set(events['anomaly_type'].unique()) - set(test_window['anomaly_type'].unique())
print(f"\nJenis yang TIDAK muncul di test window: {missing_types if missing_types else 'Tidak ada'}")

      event_id  day_start  day_end  bus_id        anomaly_type  n_rows
29          30          4        5  BUS_01          kecelakaan       7
166        167         24       25  BUS_01            overheat      19
770        771         26       27  BUS_04  sensor_malfunction       8
866        867          9       10  BUS_05               mogok      53
1005      1006          1        2  BUS_06               mogok      53
1041      1042          6        7  BUS_06          kecelakaan       7
1198      1199          2        3  BUS_07            overheat      30
1204      1205          3        4  BUS_07               mogok      58
1265      1266         11       12  BUS_07          kecelakaan       3
1392      1393          1        2  BUS_08            overheat      15
1571      1572         28       29  BUS_08               mogok      47
1605      1606          3        4  BUS_09           overspeed      15
1631      1632          6        7  BUS_09          overloaded      18
1795  

In [ ]:
# Cek event yang nyebrang persis di cutoff yang dipilih
boundary_events = events[(events['day_start'] < 25) & (events['day_end'] >= 25)]
print(boundary_events)  

      event_id  day_start  day_end  bus_id anomaly_type  n_rows
166        167         24       25  BUS_01     overheat      19
4737      4738         24       25  BUS_24        mogok      40


In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve, auc, classification_report

DATASET_PATH = "../data/bus_anomaly_dataset.csv"
FEATURE_COLS = ["speed_kmh", "passenger_count", "engine_temp_c", "rolling_mean_1h", "z_score"]

df = pd.read_csv(DATASET_PATH)
df = df.sort_values(["bus_id", "day", "timestamp"]).reset_index(drop=True)

# Exclude 59 baris dari 2 event yang nyebrang boundary day 24/25
# (BUS_01 overheat idx 167-185, BUS_24 mogok idx 4738-4777 — sesuaikan kalau index berubah setelah sort)
boundary_event_mask = (
    ((df["bus_id"] == "BUS_01") & (df["anomaly_type"] == "overheat") & (df["day"].isin([24, 25])))
    | ((df["bus_id"] == "BUS_24") & (df["anomaly_type"] == "mogok") & (df["day"].isin([24, 25])))
)
# Baris boundary day=24 dari 2 event ini dibuang total (tidak masuk train maupun test)
df = df[~(boundary_event_mask & (df["day"] == 24))].reset_index(drop=True)

#  Time-based split 
train_df = df[df["day"] <= 24].copy()
test_df = df[df["day"] >= 25].copy()

print(f"Train: {len(train_df):,} baris | Test: {len(test_df):,} baris")
print(f"Anomaly rate train: {train_df['is_anomaly'].mean():.4f} | test: {test_df['is_anomaly'].mean():.4f}")

#  Training Isolation Forest (semi-supervised: fit di seluruh train termasuk anomali, sesuai pendekatan kamu) 
X_train = train_df[FEATURE_COLS]
X_test = test_df[FEATURE_COLS]
y_test = test_df["is_anomaly"]

iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_train)

# score_samples: makin rendah/negatif = makin anomali. Invert supaya makin tinggi = makin anomali (lebih intuitif buat PR-AUC)
anomaly_score = -iso.score_samples(X_test)
y_pred = iso.predict(X_test)
y_pred_binary = (y_pred == -1).astype(int)

#  Evaluasi 
precision, recall, _ = precision_recall_curve(y_test, anomaly_score)
pr_auc = auc(recall, precision)
print(f"\nPR-AUC (test set, day 25-30): {pr_auc:.4f}")

print("\nClassification Report (threshold default IsolationForest):")
print(classification_report(y_test, y_pred_binary, target_names=["Normal", "Anomaly"]))

Train: 1,468,631 baris | Test: 367,200 baris
Anomaly rate train: 0.0506 | test: 0.0470

PR-AUC (test set, day 25-30): 0.7932

Classification Report (threshold default IsolationForest):
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99    349937
     Anomaly       0.72      0.73      0.72     17263

    accuracy                           0.97    367200
   macro avg       0.85      0.86      0.86    367200
weighted avg       0.97      0.97      0.97    367200



In [ ]:
import joblib
import json
import numpy as np

#  Dump model 
joblib.dump(iso, "model3_anomaly_detector_v2.pkl")

#  Hitung severity thresholds dari train set yang baru 
train_scores = -iso.score_samples(X_train)  # makin tinggi = makin anomali, konsisten sama evaluasi tadi

severity_thresholds = {
    "High": float(np.percentile(train_scores, 99)),     # p99 — paling ekstrem
    "Medium": float(np.percentile(train_scores, 97)),   # p97 — sesuai gating 
    "Low": float(np.percentile(train_scores, 95)),       # p95
}

#  Config baru 
config = {
    "contamination": 0.05,
    "features": ["speed_kmh", "passenger_count", "engine_temp_c", "rolling_mean_1h", "z_score"],
    "anomaly_label_mapping": {
        "IF_output_-1": 1,
        "IF_output_1": 0
    },
    "severity_thresholds": severity_thresholds,
    "score_direction_note": "score = -iso.score_samples(X); makin tinggi makin anomali",
    "train_test_split": {
        "method": "time-based, event-aware",
        "train_days": "1-24",
        "test_days": "25-30",
        "excluded_boundary_rows": 59
    },
    "scaler": None,
    "model": "model3_anomaly_detector_v2.pkl"
}

with open("config_model3_v2.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))

{
  "contamination": 0.05,
  "features": [
    "speed_kmh",
    "passenger_count",
    "engine_temp_c",
    "rolling_mean_1h",
    "z_score"
  ],
  "anomaly_label_mapping": {
    "IF_output_-1": 1,
    "IF_output_1": 0
  },
  "severity_thresholds": {
    "High": 0.631770500923413,
    "Medium": 0.5904288660154836,
    "Low": 0.5700762350811471
  },
  "score_direction_note": "score = -iso.score_samples(X); makin tinggi makin anomali",
  "train_test_split": {
    "method": "time-based, event-aware",
    "train_days": "1-24",
    "test_days": "25-30",
    "excluded_boundary_rows": 59
  },
  "scaler": null,
  "model": "model3_anomaly_detector_v2.pkl"
}
